# Flood summaries for ThinkHazard

This script performs flood hazard ranking by administrative unit using global-extent
Fathom tiles hosted on AWS S3, rather than country-extent locally downloaded data.

The hazard ranking is based on:
- Value threshold: Minimum flood depth (cm) to consider
- Area threshold: Minimum percentage of area affected
- Hazard score: Count of return periods meeting both thresholds (0-3)


In [1]:
import os, time, io, json, sys
import urllib3
import boto3

import geopandas as gpd
import pandas as pd
import numpy as np

from functools import reduce
from urllib3.exceptions import InsecureRequestWarning
from botocore import UNSIGNED
from botocore.config import Config
from tqdm.notebook import tqdm

# Import helper functions
from gfdrr_helper import *

urllib3.disable_warnings(InsecureRequestWarning)

def tPrint(s):
    """prints the time along with the message"""
    print("%s\t%s" % (time.strftime("%H:%M:%S"), s))

s3_client = boto3.client('s3', verify=False, config=Config(signature_version=UNSIGNED))

%load_ext autoreload
%autoreload 2

In [2]:
local_folder = "C:/WBG/Work/Projects/ThinkHazard"
out_folder = os.path.join(local_folder, "FATHOM_summaries")
map_folder = os.path.join(local_folder, "FATHOM_maps")
for tF in [out_folder, map_folder]:
    if not os.path.exists(tF):
        os.makedirs(tF)
vrt_folder = r"C:\WBG\Work\data\FATHOM"
s3_bucket = "wbg-geography01"
s3_prefix = "FATHOM"
return_periods = [10, 100, 500, 1000]
flood_files = [
    ["FU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-FLUVIAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ["CU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-COASTAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ['PD', "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-PLUVIAL-DEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"]
]

admin_boundaries_file = r"C:\WBG\Work\data\ADMIN\NEW_WB_BOUNDS\FOR_PUBLICATION\crs_4326\parquet\WB_GAD_ADM2.parquet"
inA = gpd.read_parquet(admin_boundaries_file)

In [ ]:
cur_out_folder = os.path.join(out_folder, "FATHOM_Detailed")
if not os.path.exists(cur_out_folder):
    os.makedirs(cur_out_folder)

cur_map_folder = os.path.join(map_folder, "FATHOM_Detailed")
if not os.path.exists(cur_map_folder):
    os.makedirs(cur_map_folder)

with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
    for sel_country in inA['ISO_A3'].unique():
        all_res = []
        out_file = os.path.join(cur_out_folder, f"FATHOM_ThinkHazard_summary_{sel_country}.csv")
        sel_a = inA[inA['ISO_A3'] == sel_country]                           
        if not os.path.exists(out_file) and not (sel_country in ["FJI",'RUS']):
            tPrint(f"Processing country: {sel_country}")
            for lbl, raster_file in flood_files:
                for return_period in return_periods:
                    tPrint(f"Processing {lbl} for {return_period} year return period")
                    sel_raster_file = raster_file.format(rp=return_period)
                    sel_raster = f"s3://{s3_bucket}/{s3_prefix}/{sel_raster_file}"
                    res_a = calculate_think_hazard_score(sel_a, sel_raster, 
                                                         depth_threshold=50, idx_col='ADM2CD_c',
                                                         all_touched=True, min_val=0, max_val=10000, no_data=-32768)
                    res_a.rename(columns={'frac_area_flooded': f'frac_area_flooded_{lbl}_{return_period}yr',
                                          #'area_flooded': f'area_flooded_{lbl}_{return_period}yr',
                                          #'total_area': f'total_area_{lbl}_{return_period}yr'                                          
                                          }, inplace=True)
                    all_res.append(res_a)                    
            all_res_df = reduce(lambda left, right: pd.merge(left, right, on='ADM2CD_c', how='outer'), all_res) 
            all_res_df.to_csv(out_file, index=False)

            sel_a = inA[inA['ISO_A3'] == sel_country]                                   
            map_adm = pd.merge(sel_a, all_res_df, on='ADM2CD_c', how='left')
            map_flood(map_adm, return_period=100, out_file=os.path.join(cur_map_folder, f"flood_map_{sel_country}_100yr.png"))
        else:
            tPrint(f"File already exists for {sel_country}, skipping...")

15:46:24	File already exists for AUS, skipping...
15:46:24	File already exists for BRN, skipping...
15:46:24	File already exists for GUM, skipping...
15:46:24	File already exists for IDN, skipping...
15:46:24	File already exists for JPN, skipping...
15:46:24	File already exists for KOR, skipping...
15:46:24	File already exists for MNP, skipping...
15:46:24	File already exists for MYS, skipping...
15:46:24	File already exists for NZL, skipping...
15:46:24	File already exists for PHL, skipping...
15:46:24	File already exists for PLW, skipping...
15:46:24	File already exists for PNG, skipping...
15:46:24	File already exists for PRK, skipping...
15:46:24	File already exists for TLS, skipping...
15:46:24	File already exists for UMI, skipping...
15:46:24	File already exists for VNM, skipping...
15:46:24	File already exists for CHN, skipping...
15:46:24	File already exists for FJI, skipping...
15:46:24	File already exists for MAC, skipping...
15:46:24	File already exists for NCL, skipping...


# DEBUGGING

In [59]:
sel_country = 'TUN'
sel_a = inA[inA['ISO_A3'] == sel_country]         
row = sel_a.loc[sel_a['ADM2CD_c'] == 'TUN012009'].iloc[0]                
raster_path = f"s3://{s3_bucket}/{s3_prefix}/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in100-COASTAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"
nodata_value = -32768
all_touched=False
depth_threshold = 50

with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
    curRaster = rasterio.open(raster_path)

    geometry = row["geometry"]    
    ul = curRaster.index(*geometry.bounds[0:2])
    lr = curRaster.index(*geometry.bounds[2:4])
    # read the subset of the data into a numpy array
    window = (
        (float(lr[0]), float(ul[0] + 1)),
        (float(ul[1]), float(lr[1] + 1)),
    )

    data = curRaster.read(1, window=window)
    data = np.where(data == nodata_value, np.nan, data)

    t = curRaster.transform
    shifted_affine = Affine(
        t.a, t.b, t.c + ul[1] * t.a, t.d, t.e, t.f + lr[0] * t.e
    )

    # rasterize the geometry
    mask = rasterize(
        [(geometry, 0)],
        out_shape=data.shape,
        transform=shifted_affine,
        fill=1,
        all_touched=all_touched,
        dtype=np.uint8,
    )

    # Add to the mask areas that are nan in the data
    mask = np.where(np.isnan(data), 1, mask)

    # create a masked numpy array
    masked_data = np.ma.array(data=data, mask=mask.astype(bool))



In [60]:
area_flooded = (masked_data > depth_threshold).sum()

In [61]:
masked_data.count()

167904

In [39]:
np.nanmax(masked_data)

87.0

In [40]:
masked_data.count()

209944

In [41]:
np.count_nonzero(masked_data)

146584

In [44]:
area_flooded = (masked_data > 50).sum()
area_flooded

1116

In [48]:
np.count_nonzero(~np.isnan(masked_data))

318137

In [49]:
masked_data.count()

209944

In [50]:
mask

array([[1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1],
       ...,
       [1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1]], dtype=uint8)

In [51]:
mask.astype(bool)

array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       ...,
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True]])

array([[ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       ...,
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True]])

In [57]:
data

array([[nan, nan, nan, ...,  0.,  0.,  0.],
       [nan, nan, nan, ...,  0.,  0.,  0.],
       [nan, nan, nan, ...,  0.,  0.,  0.],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])